
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


## Data Generation

This notebook will set up prerequisite tables and perform the `CLUSTER BY` operation. It is recommended that you use a large cluster with a minimum of 4 workers to run this notebook in a shorter amount of time.

In [0]:
%sql
USE CATALOG hive_metastore

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbacademy

In [0]:
%sql
USE SCHEMA dbacademy

In [0]:
df = spark.read.csv('/databricks-datasets/airlines/part-00000', inferSchema=True, header=True)

In [0]:
df2 = spark.read.csv('/databricks-datasets/airlines/', schema=df.schema, header=True)

In [0]:
from pyspark.sql.functions import *

(df2
 .withColumn('id', monotonically_increasing_id())
 .select('id', 'year', 'FlightNum', 'ArrDelay', 'UniqueCarrier', 'TailNum')
 .write
 .mode("overwrite")
 .saveAsTable('dbacademy.flights')
)

In [0]:
%sql
OPTIMIZE dbacademy.flights ZORDER BY FlightNum

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.flights_cluster_id;
CREATE TABLE dbacademy.flights_cluster_id CLUSTER BY (id) AS SELECT * FROM dbacademy.flights

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.flights_cluster_id_flightnum;
CREATE TABLE dbacademy.flights_cluster_id_flightnum
  CLUSTER BY (id, FlightNum)
  AS SELECT * FROM dbacademy.flights

In [0]:
df_small = spark.read.csv('/databricks-datasets/airlines/part-000*', schema=df.schema, header=True)
df_small.filter('year = 1998 or year = 1999').write.mode("overwrite").saveAsTable('dbacademy.flights_small')

In [0]:
%sql
DESCRIBE DETAIL dbacademy.flights

In [0]:
%sql
DESCRIBE DETAIL dbacademy.flights_cluster_id

In [0]:
%sql
DESCRIBE DETAIL dbacademy.flights_cluster_id_flightnum


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>